# Walk-forward demo

Offline fixture, generic EMA-cross candidates, rolling train/validation windows.
The lesson is process: compare in-sample and out-of-sample behavior before trusting a signal.

In [ ]:
from pathlib import Path

import pandas as pd

from confscan.backtest.engine import run_backtest
from confscan.backtest.metrics import Metrics
from confscan.backtest.walk_forward import classify_robustness, walk_forward_split
from confscan.signals.ta import detect_cross, ema

df = pd.read_csv(Path("tests/fixtures/btc_4h_sample.csv"), parse_dates=["timestamp"]).set_index(
    "timestamp"
)
candidates = {"fast": (10, 24), "textbook": (12, 26), "smooth": (18, 36)}
rows = []
for fold_no, fold in enumerate(
    walk_forward_split(df, train_days=20, val_days=5, step_days=5), start=1
):
    train_scores = {}
    val_scores = {}
    for name, (fast, slow) in candidates.items():
        tr_cross = detect_cross(ema(fold.train["close"], fast), ema(fold.train["close"], slow))
        va_cross = detect_cross(ema(fold.val["close"], fast), ema(fold.val["close"], slow))
        train_scores[name] = Metrics.from_result(
            run_backtest(fold.train, tr_cross == 1, tr_cross == -1)
        ).sharpe
        val_scores[name] = Metrics.from_result(
            run_backtest(fold.val, va_cross == 1, va_cross == -1)
        ).sharpe
    train_rank = {
        k: i + 1
        for i, (k, _) in enumerate(sorted(train_scores.items(), key=lambda x: x[1], reverse=True))
    }
    val_rank = {
        k: i + 1
        for i, (k, _) in enumerate(sorted(val_scores.items(), key=lambda x: x[1], reverse=True))
    }
    best = min(train_rank, key=train_rank.get)
    rows.append(
        {
            "fold": fold_no,
            "candidate": best,
            "train_rank": train_rank[best],
            "val_rank": val_rank[best],
            "verdict": classify_robustness(train_rank[best], val_rank[best], len(candidates)),
        }
    )
pd.DataFrame(rows)